<center><h1>Hu_Tianyang_HW3</h1></center>
<br>
<br>

Name: Tianyang Hu
<br>
Github Username: hhuhhuhhuhhu
<br>
USC ID: 2203816542

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [2]:
import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib.pyplot as plt
import os
import zipfile
import urllib.request
from pathlib import Path

Get the AReM Data Set

In [3]:
url = "https://archive.ics.uci.edu/static/public/366/activity+recognition+system+based+on+multisensor+data+fusion+arem.zip"

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

zip_path = data_dir / "AReM.zip"

if not zip_path.exists():
    urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(data_dir)

for item in sorted(data_dir.iterdir()):
    print(item.name)

AReM.zip
bending1
bending2
bendingType.pdf
cycling
lying
sensorsPlacement.pdf
sitting
standing
walking


### (b) Test and Train Data

In [5]:
activities = ["bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking"]

train_files = []
test_files = []

for activity in activities:
    folder = data_dir / activity
    files = sorted(folder.glob("dataset*.csv"), key=lambda x: int(x.stem.replace("dataset", "")))

    for file in files:
        number = int(file.stem.replace("dataset", ""))

        if activity in ["bending1", "bending2"]:
            if number <= 2:
                test_files.append(file)
            else:
                train_files.append(file)
        else:
            if number <= 3:
                test_files.append(file)
            else:
                train_files.append(file)

### (c) Feature Extraction

#### i. Research

minimum, maximum, mean, median, standard deviation, range, variance, first quartile, third quartile

#### ii. Extraction

In [16]:
features = ["min", "max", "mean", "median", "std", "1st quart", "3rd quart"]
rows = []

for activity in activities:
    folder = data_dir / activity
    files = sorted(folder.glob("dataset*.csv"), key=lambda x: int(x.stem.replace("dataset", "")))

    for file in files:
        with open(file, "r") as f:
            lines = [line.strip().rstrip(",") for line in f if line.strip() and not line.startswith("#")]

        df = pd.DataFrame([line.replace(",", " ").split() for line in lines])
        df = df.iloc[:, 1:7].apply(pd.to_numeric, errors="coerce")

        row = {"Instance": f"{activity}/{file.name}"}

        for i in range(6):
            series = df.iloc[:, i].dropna()

            row[f"min{i+1}"] = series.min()
            row[f"max{i+1}"] = series.max()
            row[f"mean{i+1}"] = series.mean()
            row[f"median{i+1}"] = series.median()
            row[f"std{i+1}"] = series.std()
            row[f"1st quart{i+1}"] = series.quantile(0.25)
            row[f"3rd quart{i+1}"] = series.quantile(0.75)

        rows.append(row)

feature_df = pd.DataFrame(rows)
display(feature_df)

,Instance,min1,max1,mean1,median1,std1,1st quart1,3rd quart1,min2,max2,...,std5,1st quart5,3rd quart5,min6,max6,mean6,median6,std6,1st quart6,3rd quart6
0,bending1/dataset1.csv,37.25,45.00,40.624792,40.50,1.476967,39.25,42.0000,0.0,1.30,...,2.188449,33.0000,36.00,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000
1,bending1/dataset2.csv,38.00,45.67,42.812812,42.50,1.435550,42.00,43.6700,0.0,1.22,...,1.995255,32.0000,34.50,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000
2,bending1/dataset3.csv,35.00,47.40,43.954500,44.33,1.558835,43.00,45.0000,0.0,1.70,...,1.999604,35.3625,36.50,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400
3,bending1/dataset4.csv,33.00,47.75,42.179812,43.50,3.670666,39.15,45.0000,0.0,3.00,...,3.849448,30.4575,36.33,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000
4,bending1/dataset5.csv,33.00,45.75,41.678063,41.75,2.243490,41.33,42.7500,0.0,2.83,...,2.411026,28.4575,31.25,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,walking/dataset11.csv,19.50,45.33,33.586875,34.25,4.650935,30.25,37.0000,0.0,14.67,...,3.283983,13.7300,18.25,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225
84,walking/dataset12.csv,19.75,45.50,34.322750,35.25,4.752477,31.00,38.0000,0.0,13.47,...,3.119856,13.5000,17.75,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650
85,walking/dataset13.csv,19.50,46.00,34.546229,35.25,4.842294,31.25,37.8125,0.0,12.47,...,2.823124,14.0000,17.75,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350
86,walking/dataset14.csv,23.50,46.25,34.873229,35.25,4.531720,31.75,38.2500,0.0,14.82,...,3.131076,13.7500,18.00,0.00,9.51,3.424646,3.27,1.690960,2.1700,4.5000


#### iii. Standard Deviation

In [18]:
from scipy.stats import bootstrap

feature_data = feature_df.drop(columns=["Instance"])

results = []

for column in feature_data.columns:
    values = feature_data[column].dropna().to_numpy()

    std = np.std(values, ddof=1)

    ci = bootstrap(
        (values,),
        np.std,
        confidence_level=0.90,
        n_resamples=10000,
        method="percentile",
        random_state=42
    ).confidence_interval

    results.append({
        "Feature": column,
        "Standard Deviation": std,
        "CI Lower": ci.low,
        "CI Upper": ci.high
    })

std_df = pd.DataFrame(results)
display(std_df)

,Feature,Standard Deviation,CI Lower,CI Upper
0,min1,9.569975,8.220066,10.720371
1,max1,4.394362,3.331748,5.269701
2,mean1,5.335718,4.677475,5.855842
3,median1,5.440054,4.759581,5.974944
4,std1,1.772153,1.556183,1.939205
5,1st quart1,6.153590,5.522698,6.615684
6,3rd quart1,5.138925,4.305528,5.811481
7,min2,0.000000,0.000000,0.000000
8,max2,5.062729,4.595717,5.371528
9,mean2,1.574164,1.389031,1.696792


#### iv. Select Features

mean, maximum and standard deviation

## 2. ISLR 3.7.4

### (a) Linear Train

The cubic regression should be lower than the linear one since the cubic one has already included the linear one and but also other situations.

### (b) Linear Test

The linear regression should have a lower test RSS since the true relationship is linear while cubic regression tends to overfit.

### (c) Not Linear Train

The cubic regression should have a lower train RSS since it can fit more situations.

### (d) Not Linear Testing

The linear regression should have a lower test RSS since it tends to have a lower variance.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 